[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# *IAM*, *OIDC* y autenticación sin llaves.

Este cuaderno explica por qué las credenciales estáticas son un riesgo estructural en *pipelines* de *CI/CD* y cómo *OpenID Connect* (*OIDC*) resuelve ese problema mediante identidad federada y efímera.

## Objetivos.

- Comprender el riesgo concreto de las llaves estáticas de servicio (*JSON keys*).
- Entender el protocolo *OIDC* y el rol del token *JWT* en la autenticación federada.
- Describir el flujo completo: *GitHub Actions* → *GCP Security Token Service* → acceso acotado.
- Configurar *Workload Identity Federation* como mecanismo de confianza verificable.
- Relacionar *OIDC* con el modelo de mínimo privilegio del curso.

## El riesgo: credenciales estáticas en *pipelines*.

En la *NB01* se documentaron dos incidentes directamente relacionados con este tema:

- **Codecov (2021):** un atacante modificó el *bash uploader* oficial para exfiltrar variables de entorno del *CI* — entre ellas llaves de servicio y tokens de acceso. Cada organización que ejecutó ese *script* expuso sus credenciales sin saberlo.
- **tj-actions (2025):** una *GitHub Action* comprometida imprimió en los *logs* todos los secretos disponibles en el *runner*, incluyendo llaves estáticas de *GCP*, *AWS* y *Azure* de los repositorios afectados.

En ambos casos el mecanismo de daño fue el mismo: **una credencial de larga duración almacenada como secreto fue exfiltrada y pudo reutilizarse fuera del *pipeline* original**.

### Por qué las *JSON keys* son un riesgo estructural.

Un *service account JSON key* de *GCP* es un archivo que:

- **No expira** a menos que se revoque manualmente.
- **No deja rastro** al copiarse — el atacante tiene una copia idéntica al original.
- **Funciona desde cualquier máquina**, sin importar el origen.
- Suele ser **sobre-privilegiado** porque asignar permisos exactos por *job* resulta tedioso.
- Requiere **rotación coordinada**: actualizar el secreto en *GitHub* y en todos los sistemas que lo consumen.

El resultado es un modelo donde un secreto de texto plano, con permisos amplios y vida indefinida, circula por sistemas de *CI/CD* — exactamente el patrón que estos incidentes explotan.

## Credencial estática vs. identidad efímera.

| Propiedad | *JSON key* estática | Token *OIDC* efímero |
|---|---|---|
| Duración | Indefinida (años) | 5–10 minutos |
| Revocación | Manual, coordinada | No necesaria: expira solo |
| Portabilidad | Funciona en cualquier máquina | Solo válido para el *job* que lo solicitó |
| Rastro de uso | No | Sí: *logs* de *Cloud STS* |
| Privilegios | Habitualmente amplios | Acotados al *job* concreto |
| Exposición en *logs* | Fatal (texto plano reutilizable) | Inocua: el token ya expiró |

El objetivo de este módulo es pasar de la columna izquierda a la derecha sin perder funcionalidad.

## *OpenID Connect* (*OIDC*).

*OpenID Connect* es una capa de identidad construida sobre *OAuth 2.0*. Mientras *OAuth 2.0* responde "¿qué puede hacer este cliente?", *OIDC* responde "¿quién es este cliente?".

### Actores en el flujo de federación.

En el contexto de *GitHub Actions* + *GCP* intervienen tres actores:

| Actor | Rol | En este curso |
|---|---|---|
| **Identity Provider (*IdP*)** | Emite y firma tokens de identidad | *GitHub Actions* |
| **Relying Party** | Verifica y confía en los tokens del *IdP* | *GCP Security Token Service* |
| **Subject** | La entidad cuya identidad se afirma | El *job* del *workflow* |

*GitHub* puede actuar como *IdP* porque publica un *endpoint* *OIDC* conocido — `https://token.actions.githubusercontent.com` — con su clave pública. Cualquier sistema que conozca esa clave puede verificar criptográficamente que un token fue emitido por *GitHub* y no fue alterado.

*GCP* acepta tokens de *GitHub* porque se configura explícitamente para hacerlo mediante *Workload Identity Federation*. No es confianza implícita, sino confianza declarada, verificable y acotada.

## Anatomía del token *JWT*.

Un *JWT* (*JSON Web Token*) tiene tres partes separadas por puntos:

```text
header . payload . signature
```

- **Header**: algoritmo de firma (habitualmente `RS256`).
- **Payload**: los *claims* — afirmaciones sobre el *subject* y el contexto de emisión.
- **Signature**: prueba criptográfica de que el token fue emitido por *GitHub* y no fue alterado.

### *Claims* del token de *GitHub Actions*.

| *Claim* | Descripción | Ejemplo |
|---|---|---|
| `iss` | Emisor | `https://token.actions.githubusercontent.com` |
| `sub` | Sujeto compuesto del *job* | `repo:org/repo:environment:prod` |
| `aud` | Audiencia — quién debe verificarlo | URL del *WIF provider* en *GCP* |
| `repository` | Repositorio que disparó el *workflow* | `PythonistaMX/api-github-actions-demo` |
| `environment` | Entorno de *GitHub Actions* | `prod` |
| `ref` | *Git ref* que disparó el *workflow* | `refs/heads/main` |
| `job_workflow_ref` | Archivo de *workflow* exacto | `org/repo/.github/workflows/deploy.yaml@refs/heads/main` |
| `iat` / `exp` | Emisión y expiración | Diferencia de 5 minutos |

El *claim* `sub` concentra en una sola cadena los tres elementos que definen la identidad del *job*: repositorio, tipo de referencia y entorno. Es lo que la condición de atributo en *GCP* evalúa para decidir si acepta o rechaza el token.

In [ ]:
# Muestra de los claims contenidos en un token OIDC de GitHub Actions.
# En producción el token nunca se inspecciona directamente:
# google-github-actions/auth@v2 lo gestiona de forma transparente.

payload_muestra = {
    "iss": "https://token.actions.githubusercontent.com",
    "sub": "repo:PythonistaMX/api-github-actions-demo:environment:prod",
    "aud": "https://iam.googleapis.com/projects/.../providers/github-actions",
    "repository": "PythonistaMX/api-github-actions-demo",
    "environment": "prod",
    "ref": "refs/heads/main",
    "job_workflow_ref": (
        "PythonistaMX/api-github-actions-demo"
        "/.github/workflows/envia-a-docker.yaml@refs/heads/main"
    ),
    "workflow": "envia-a-docker",
    "iat": 1_716_000_000,
    "exp": 1_716_000_300,
}

print("Claims del token OIDC emitido por GitHub Actions")
print("=" * 56)
for claim, valor in payload_muestra.items():
    print(f"  {claim:<25} {valor}")

duracion = payload_muestra["exp"] - payload_muestra["iat"]
print(f"\nDuración del token: {duracion}s ({duracion // 60} min) — expira por diseño.")

## El flujo completo: de *workflow* a acceso en *GCP*.

El intercambio ocurre en cuatro pasos. Ninguno involucra credenciales de larga duración.

### Paso 1 — El *workflow* declara que necesita emitir tokens.

```yaml
permissions:
  contents: read
  id-token: write   # Habilita la solicitud de tokens OIDC al runner
```

Sin `id-token: write` el *runner* no puede solicitar tokens al *endpoint* interno de *GitHub*.

### Paso 2 — El *runner* obtiene un *JWT* firmado por *GitHub*.

El *runner* llama al *endpoint* `ACTIONS_ID_TOKEN_REQUEST_URL`. *GitHub* emite un *JWT* firmado con su clave privada que contiene los *claims* del *job* y expira en 5 minutos.

### Paso 3 — El *JWT* se intercambia por un *access token* de *GCP*.

La acción `google-github-actions/auth@v2` envía el *JWT* al *GCP Security Token Service* (*STS*):

1. *GCP STS* descarga la clave pública de *GitHub* desde su *JWKS endpoint* (`https://token.actions.githubusercontent.com/.well-known/jwks`).
2. Verifica la firma del *JWT*. Si no coincide, rechaza la solicitud.
3. Evalúa la **condición de atributo**: ¿el `repository` y el `environment` del token están en la lista de confianza declarada en *Workload Identity Federation*?
4. Si la condición se cumple, emite un *access token* de *GCP* con vida máxima de 1 hora.

### Paso 4 — El *access token* se usa para llamar a *GCP*.

El token efímero se inyecta automáticamente en las herramientas del *workflow* ([*gcloud*](https://cloud.google.com/sdk/gcloud), `docker`, `terraform`). Cuando el *job* termina, el token expira sin intervención manual.

## *Subject*, identidad federada y *principal* IAM — relación entre términos.

El cuaderno usa tres términos que parecen sinónimos pero designan capas distintas del modelo:

| Término | Capa | Definición en este curso |
|---|---|---|
| **Subject** (`sub`) | Token *OIDC* | Cadena compuesta que identifica el *job*: `repo:org/repo:environment:prod`. Es un *claim* del *JWT*, no una entidad *GCP*. |
| **Identidad federada** | *WIF pool* | Representación del *job* dentro del pool de confianza de *GCP*: `principalSet://.../attribute.environment/prod`. Existe solo durante el intercambio de tokens. |
| **Principal IAM** | *GCP IAM* | Entidad a la que *GCP* asigna roles: el *service account* `cicd-deployer-prod@...`. Es quien tiene permisos reales sobre recursos. |

El flujo conecta los tres en cadena:

```text
GitHub Actions job
│
│  emite JWT con claim:
│  sub = "repo:org/repo:environment:prod"          ← Subject (OIDC)
│
▼
GCP Security Token Service
│
│  verifica firma + evalúa attribute_condition
│  asigna identidad del pool WIF:
│  principalSet://.../attribute.environment/prod   ← Identidad federada (WIF)
│
▼
Binding en el service account
│
│  roles/iam.workloadIdentityUser
│  permite suplantar al SA:
│  cicd-deployer-prod@<proyecto>.iam.gserviceaccount.com  ← Principal IAM
│
▼
GCP (Cloud Run, Artifact Registry, Cloud SQL...)
```

La condición de atributo actúa en la segunda flecha: compara el `sub` del *JWT* con los atributos mapeados en el *WIF provider* para decidir si la identidad federada tiene permiso de convertirse en ese *principal* IAM.

## *Workload Identity Federation* en *GCP*.

*Workload Identity Federation* (*WIF*) es el mecanismo de *GCP* para aceptar tokens de *IdPs* externos. Se compone de tres recursos que deben existir antes de que cualquier *workflow* pueda autenticarse.

### 1. *Workload Identity Pool*.

Define el **perímetro de confianza**: qué sistemas externos tienen permiso de presentar tokens para obtener acceso a *GCP*.

```hcl
resource "google_iam_workload_identity_pool" "github" {
  workload_identity_pool_id = "github-pool"
  display_name              = "GitHub Actions"
}
```

### 2. *Workload Identity Pool Provider*.

Configura el *IdP* específico: la URI del emisor, el mapeo de *claims* a atributos de *GCP* y la condición de aceptación.

```hcl
resource "google_iam_workload_identity_pool_provider" "github" {
  workload_identity_pool_id          = google_iam_workload_identity_pool.github.workload_identity_pool_id
  workload_identity_pool_provider_id = "github-actions"

  oidc {
    issuer_uri = "https://token.actions.githubusercontent.com"
  }

  attribute_mapping = {
    "google.subject"        = "assertion.sub"
    "attribute.repository"  = "assertion.repository"
    "attribute.environment" = "assertion.environment"
  }

  # Restringe el pool a los repositorios exactos que pueden obtener tokens.
  # Sin esta condición, cualquier repositorio de GitHub podría solicitar
  # acceso y potencialmente suplantar al service account de CI/CD.
  attribute_condition = "attribute.repository in ['org/api-github-actions-demo', 'org/infra-repo']"
}
```

Cuando el proyecto tiene dos repositorios (app + infra), `attribute_condition` lista ambos porque los dos necesitan autenticarse contra *GCP*: el repo de la app para desplegar en *Cloud Run*, el repo de infra para ejecutar `terraform apply`.

### 3. Vinculación con el *service account*.

Autoriza a un subconjunto de identidades del *pool* a suplantar al *service account*:

```hcl
resource "google_service_account_iam_member" "wif_prod" {
  service_account_id = google_service_account.cicd_deployer.name
  role               = "roles/iam.workloadIdentityUser"
  member = "principalSet://iam.googleapis.com/${google_iam_workload_identity_pool.github.name}/attribute.environment/prod"
}
```

Este *binding* establece: solo tokens del *pool* con `attribute.environment == prod` pueden suplantar a `cicd-deployer`.

### Bindings IAM adicionales del pipeline.

Además de los tres recursos anteriores, el pipeline real requiere bindings complementarios para funcionar correctamente:

```hcl
# Permite al SA de CI/CD generar identity tokens impersonándose a sí mismo.
# gcloud auth print-identity-token no funciona con credenciales WIF directamente;
# se necesita impersonación explícita para el smoke test del deploy.
resource "google_service_account_iam_member" "cicd_self_token_creator" {
  service_account_id = google_service_account.cicd_deployer.name
  role               = "roles/iam.serviceAccountTokenCreator"
  member             = "serviceAccount:${google_service_account.cicd_deployer.email}"
}

# Permite al SA de CI/CD invocar el servicio Cloud Run para el smoke test.
resource "google_cloud_run_v2_service_iam_member" "cicd_invoker" {
  name     = google_cloud_run_v2_service.app.name
  location = var.region
  role     = "roles/run.invoker"
  member   = "serviceAccount:${google_service_account.cicd_deployer.email}"
}

# Acceso público (intencional para servicios de demo).
# Para un servicio privado: eliminar este recurso.
resource "google_cloud_run_v2_service_iam_member" "public_invoker" {
  name     = google_cloud_run_v2_service.app.name
  location = var.region
  role     = "roles/run.invoker"
  member   = "allUsers"
}
```

El binding `cicd_self_token_creator` merece especial atención: las credenciales federadas WIF permiten obtener *access tokens* de *GCP*, pero **no *identity tokens* OIDC** directamente. El smoke test del pipeline necesita un *identity token* para autenticarse contra la URL exclusiva de la revisión candidata en *Cloud Run*. La solución es que el SA de CI/CD se impersone a sí mismo:

```bash
gcloud auth print-identity-token \
  --impersonate-service-account="${{ vars.GCP_SERVICE_ACCOUNT }}"
```

### *Service accounts* y permisos IAM requeridos.

En el flujo de este curso, **test no despliega a GCP**. El entorno `test` despliega a servidor Linux (ver *NB18*), mientras que `prod` usa Cloud Run en GCP.

Permisos IAM mínimos por alcance:

| Alcance | Principal | Rol IAM | Para qué se usa |
|---|---|---|---|
| Binding sobre SA deployer | `principalSet://.../attribute.environment/prod` | `roles/iam.workloadIdentityUser` | Suplantación vía OIDC/WIF |
| Proyecto | `cicd-deployer@...` | `roles/run.admin` | Desplegar y gestionar Cloud Run |
| SA runtime (binding) | `cicd-deployer@...` | `roles/iam.serviceAccountUser` | Asignar identidad runtime al deploy |
| SA deployer (self-binding) | `cicd-deployer@...` | `roles/iam.serviceAccountTokenCreator` | Generar identity tokens para smoke test |
| Cloud Run service | `cicd-deployer@...` | `roles/run.invoker` | Invocar la URL candidata en smoke test |
| Cloud Run service | `allUsers` | `roles/run.invoker` | Acceso público (solo si el servicio es abierto) |
| Proyecto | `run-runtime@...` | `roles/cloudsql.client` | Conectarse a Cloud SQL en ejecución |
| Secreto específico | `run-runtime@...` | `roles/secretmanager.secretAccessor` | Leer DATABASE_URL en ejecución |

**Importante:** `roles/iam.workloadIdentityUser` no se concede a nivel proyecto; se otorga como *binding* sobre el *service account* objetivo para evitar sobre-privilegios.

## Las condiciones de atributo como mecanismo de acotación.

La condición de atributo es la pieza que convierte la federación en confianza **explícita, verificable y acotada** — el modelo del curso aplicado a identidad.

Sin condición de atributo, cualquier repositorio de *GitHub* podría obtener tokens para suplantar al *service account*. La condición restringe exactamente qué tokens son aceptables:

- **Solo este repositorio** — bloquea *forks* y repositorios de atacantes.
- **Solo este entorno** — aísla `prod` de `test` y de entornos no declarados.
- **Solo este archivo de *workflow*** — `job_workflow_ref` permite restringir al *workflow* exacto (control adicional opcional).

La celda siguiente simula la evaluación que *GCP STS* realiza al recibir un token.

In [ ]:
# Simulación de la condición de atributo que GCP STS evalúa al recibir un token.
# Si payload_muestra no existe (ejecución fuera de orden), se crea una muestra mínima.

if "payload_muestra" not in globals():
    payload_muestra = {
        "repository": "PythonistaMX/api-github-actions-demo",
        "environment": "prod",
        "sub": "repo:PythonistaMX/api-github-actions-demo:environment:prod",
    }
    print("Nota: se creó payload_muestra mínimo porque no estaba definido en memoria.")

def evalua_condicion(claims: dict, repo_permitido: str, entorno_permitido: str) -> tuple[bool, str]:
    repo_ok = claims.get("repository") == repo_permitido
    env_ok = claims.get("environment") == entorno_permitido
    if repo_ok and env_ok:
        return True, "token aceptado — acceso al service account cloud de prod"
    if not repo_ok:
        return False, f"repositorio '{claims.get('repository')}' no está en la lista de confianza"
    return False, f"entorno '{claims.get('environment')}' no autorizado para este service account"

casos = [
    ("PythonistaMX/api-github-actions-demo", "prod", "job legítimo de prod (ruta cloud)"),
    ("PythonistaMX/api-github-actions-demo", "test", "job de test (ruta Linux) — no debe acceder a SA cloud"),
    ("attacker/fork", "prod", "fork malicioso intentando acceso a prod"),
    ("PythonistaMX/api-github-actions-demo", "dev", "entorno no declarado en WIF"),
]

REPO = "PythonistaMX/api-github-actions-demo"
ENTORNO = "prod"

print(f"Condición: attribute.repository == '{REPO}'")
print(f"Binding:   attribute.environment == '{ENTORNO}' -> cicd-deployer-prod SA")
print("=" * 70)

for repo, env, descripcion in casos:
    token = {**payload_muestra, "repository": repo, "environment": env}
    acepta, motivo = evalua_condicion(token, REPO, ENTORNO)
    simbolo = "OK" if acepta else "X"
    print(f"\n[{simbolo}] {descripcion}")
    print(f"    repository  = {repo!r}")
    print(f"    environment = {env!r}")
    print(f"    resultado   : {motivo}")

## *OIDC* y mínimo privilegio: convergencia de controles.

*OIDC* no resuelve solo el problema de las credenciales estáticas — también hace **practicable** el principio de mínimo privilegio a escala de *job*.

Con llaves estáticas, asignar permisos distintos por *job* y por entorno requiere crear y rotar múltiples *JSON keys*. La complejidad operativa lleva invariablemente a un solo *service account* con permisos amplios "para que todo funcione".

Con *OIDC* + entornos de *GitHub Actions*, la granularidad es consecuencia directa del modelo de confianza:

| Elemento | Mecanismo de acotación |
|---|---|
| **¿Qué repositorio?** | `attribute.repository` en la condición del *WIF provider* |
| **¿Qué entorno?** | `attribute.environment` en el *binding* del *service account* |
| **¿Qué *workflow*?** | `attribute.job_workflow_ref` (restricción adicional opcional) |
| **¿Qué puede hacer?** | Roles del *service account* — mínimos por entorno |
| **¿Por cuánto tiempo?** | *JWT*: 5 min; *access token* de *GCP*: hasta 1 hora |

El resultado es el modelo de confianza del curso aplicado a identidad: **explícita** (declarada en *Terraform*), **verificable** (firmada criptográficamente) y **acotada al mínimo necesario**.

## Configuración en el *workflow*.

El *job* que necesita autenticarse contra *GCP* requiere dos cambios respecto a un *job* estándar:

```yaml
permissions:
  contents: read
  id-token: write           # Autoriza la emisión del token OIDC al runner

jobs:
  deploy:
    runs-on: ubuntu-latest
    environment: prod       # Define qué service account aplica vía WIF binding

    steps:
      - uses: actions/checkout@<SHA>

      - name: Autenticar en GCP (OIDC — sin llaves estáticas)
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ vars.GCP_WORKLOAD_IDENTITY_PROVIDER }}
          service_account:            ${{ vars.GCP_SERVICE_ACCOUNT }}
```

`GCP_WORKLOAD_IDENTITY_PROVIDER` tiene la forma:

```text
projects/<número>/locations/global/workloadIdentityPools/<pool>/providers/<provider>
```

Este valor **no es un secreto** — es un identificador de recurso de *GCP*. Se guarda como *variable* (no *secret*) en el entorno de *GitHub Actions*. Lo mismo aplica a `GCP_SERVICE_ACCOUNT`.

El *workflow* de referencia del curso que implementa este patrón completo es `.github/workflows/terraform-plan-apply.yml` en el submódulo `infra/terraform-py271`.

In [ ]:
# Permisos IAM mínimos por identidad (modelo explícito para este curso).
# test se despliega a Linux (NB18), prod se despliega a GCP.

sa_deployer_prod = "cicd-deployer-prod@<proyecto>.iam.gserviceaccount.com"
sa_runtime = "run-runtime@<proyecto>.iam.gserviceaccount.com"
target_test = "Servidor Linux (NB18)"

permisos_deployer_prod = [
    ("roles/run.admin", "Proyecto", "Desplegar y gestionar revisiones/tráfico en Cloud Run."),
    (
        "roles/iam.serviceAccountUser",
        f"Service Account {sa_runtime}",
        "Asignar la identidad runtime al servicio durante el deploy.",
    ),
]

binding_wif_prod = (
    "roles/iam.workloadIdentityUser",
    "Binding sobre SA deployer prod",
    "Permite que el principalSet del pool WIF suplante el SA de prod sin llaves JSON.",
)

permisos_runtime = [
    ("roles/secretmanager.secretAccessor", "Proyecto o secreto", "Leer secretos en ejecución."),
    ("roles/cloudsql.client", "Proyecto", "Conectarse a Cloud SQL en ejecución (si aplica)."),
]

print("Arquitectura de identidades")
print("=" * 72)
print(f"  Entorno test      : {target_test}")
print(f"  SA deployer prod  : {sa_deployer_prod}")
print(f"  SA runtime (prod) : {sa_runtime}")

print("\nPermisos del SA deployer de prod")
print("-" * 72)
for rol, alcance, uso in permisos_deployer_prod:
    print(f"  {rol:<30} | {alcance:<34} | {uso}")

rol, alcance, uso = binding_wif_prod
print("\nBinding WIF obligatorio (prod)")
print("-" * 72)
print(f"  {rol:<30} | {alcance:<34} | {uso}")

print("\nPermisos del SA runtime (prod)")
print("-" * 72)
for rol, alcance, uso in permisos_runtime:
    print(f"  {rol:<30} | {alcance:<34} | {uso}")

print("\nPrincipio clave: WIF para prod cloud; test Linux fuera de WIF GCP.")

## Riesgos que mitiga *OIDC*.

| Riesgo | Vector de ataque | Cómo lo mitiga *OIDC* |
|---|---|---|
| **Exfiltración de credenciales** | Acción comprometida imprime secretos (tj-actions) | No hay credencial estática que exfiltrar |
| **Reutilización fuera del *pipeline*** | Token robado usado desde la máquina del atacante | El *JWT* expira en 5 min; el atacante no llega a tiempo |
| **Acceso persistente tras compromiso** | Clave copiada sigue activa meses después | Sin rotación necesaria; cada emisión es nueva y efímera |
| **Sobre-privilegios acumulados** | SA con permisos amplios "para que todo funcione" | Los roles se asignan por entorno y *job* específico |
| **Escalada lateral entre entornos** | Clave de `test` usada para acceder a `prod` | Los *bindings* de *WIF* son independientes por entorno |
| **Acceso desde *forks* no autorizados** | *Fork* del repositorio obtiene acceso a producción | `attribute.repository` restringe al repositorio exacto |
| **Uso desde *workflows* no autorizados** | *Workflow* no oficial en el mismo repositorio | `attribute.job_workflow_ref` permite acotar al archivo exacto |

### Limitaciones.

*OIDC* no elimina todos los riesgos del *pipeline*:

- Un *job* comprometido puede usar el *access token* durante su vida útil (hasta 1 hora).
- Los roles del *service account* deben ser correctamente acotados — *OIDC* facilita el mínimo privilegio pero no lo garantiza.
- La condición de atributo debe auditarse: una condición demasiado permisiva (p. ej., `attribute.repository.startsWith('org/')`) amplía el radio de *blast* involuntariamente.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>